# Fine-tuning LoRA — Assistant IT support
**Brief 6.1 — Cisia Module 3 — Concevoir et iMplémenter une Solution d'IA**

Ce notebook est la version **commentée** du notebook de démonstration fourni.
Il met l'accent sur les trois briques clés du pipeline PEFT/TRL :

1. `LoraConfig`  — adaptation low-rank du modèle.
2. `SFTConfig`   — hyperparamètres de l'entraînement supervisé.
3. `SFTTrainer`  — orchestration de l'entraînement et **masquage de la loss sur le prompt**.

Il charge `gold_data.jsonl` si disponible (sortie de `gold_data.py`),
sinon il bascule sur un mini jeu d'exemples métier codé en dur.


## 1. Installation (à exécuter une seule fois)

In [1]:
%pip install -q transformers datasets peft trl accelerate
# Décommenter si exécution hors environnement Cisia.

Note: you may need to restart the kernel to use updated packages.


## 2. Imports

In [2]:
import json, os
from pathlib import Path

import torch
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", DEVICE)

Device : cpu


## 3. Configuration globale

In [3]:
MODEL_NAME       = "HuggingFaceTB/SmolLM2-135M"
GOLD_PATH        = Path("gold_data.jsonl")
OUTPUT_DIR       = "./it_assistant_finetuned"
QUESTION_TEST    = "Comment réinitialiser mon mot de passe Windows ?"
QUESTION_GENERAL = "Quelle est la procédure pour signaler un incident sécurité ?" 

## 4. Données métier
On privilégie `gold_data.jsonl` (sortie de l'étape 3 du pipeline).
À défaut, on utilise un fallback minimal pour pouvoir exécuter le notebook
de bout en bout en démo.

In [ ]:
EXEMPLES_METIER = [
    {"instruction": "Comment réinitialiser mon mot de passe ?",
     "reponse":     "Rendez-vous sur AtosConnect → « Mot de passe oublié », "
                    "validez via le SMS reçu sur votre téléphone professionnel."},
    {"instruction": "Mon ordinateur ne démarre plus.",
     "reponse":     "Ouvrez un ticket sur AtosConnect (catégorie « Poste de travail »), "
                    "précisez le numéro de série et joignez une photo de l'écran."},
    {"instruction": "Je n'ai plus accès à ma boîte mail.",
     "reponse":     "Vérifiez votre connexion VPN, puis ouvrez un ticket "
                    "sur AtosConnect catégorie « Messagerie »."},
]

if GOLD_PATH.exists():
    raw_ds = load_dataset("json", data_files=str(GOLD_PATH), split="train")
    print(f"Dataset chargé depuis {GOLD_PATH} : {len(raw_ds)} exemples")
else:
    raw_ds = Dataset.from_list(EXEMPLES_METIER)
    print(f"Fallback : {len(raw_ds)} exemples codés en dur")

## 5. Prompt template & formatter
Le template `### Instruction / ### Réponse` est volontairement simple :
il sert de marqueur clair pour que le modèle apprenne où commence la
*réponse* (= zone sur laquelle la loss sera calculée).

In [4]:
PROMPT_TEMPLATE = (
    "### Instruction\n"
    "{instruction}\n\n"
    "### Réponse\n"
    "{reponse}"
)

def formatter(example):
    return PROMPT_TEMPLATE.format(
        instruction=example["instruction"],
        reponse=example["reponse"],
    )

ds = raw_ds.map(
    lambda ex: {"text": formatter(ex)},
    remove_columns=[c for c in raw_ds.column_names if c != "text"],
)
print(ds[0]["text"])

NameError: name 'raw_ds' is not defined

## 6. Chargement du modèle de base

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
print(f"Modèle {MODEL_NAME} chargé ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)")

## 7. Inférence AVANT fine-tuning (baseline)

In [ ]:
def generate(model, question, max_new_tokens=120):
    prompt = PROMPT_TEMPLATE.format(instruction=question, reponse="").rstrip()
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(generate(model, QUESTION_TEST))

## 8. `LoraConfig` — l'adaptation low-rank en détail

`LoraConfig` (librairie **PEFT**) gèle entièrement les poids du modèle de base
et n'entraîne **que** deux petites matrices `A` (down-projection) et `B`
(up-projection) injectées en parallèle de certaines couches linéaires.

| Paramètre | Valeur retenue | Rôle |
|---|---|---|
| `r` | **16** | Rang des matrices `A` et `B`. Plus `r` est grand, plus l'adapter a de capacité — mais aussi de paramètres à entraîner. `16` est un bon compromis pour un petit jeu de données métier. |
| `lora_alpha` | **32** | Facteur de scaling. La sortie LoRA est multipliée par `lora_alpha / r` (= 2 ici). Un ratio agressif (`>=2`) accélère la convergence sur petit dataset. |
| `lora_dropout` | **0.05** | Léger dropout pour limiter le sur-apprentissage. |
| `target_modules` | `["q_proj","v_proj"]` | Couches d'attention `query` et `value` — les plus impactantes pour adapter le **style** et le **vocabulaire métier**. |
| `bias` | `"none"` | Aucune mise à jour des biais → moins de paramètres entraînés, sobriété mémoire. |
| `task_type` | `CAUSAL_LM` | Mode auto-régressif (prédiction du prochain token). |

Conséquence : on entraîne typiquement **< 1 %** des paramètres totaux du modèle,
ce qui permet de respecter la contrainte de sobriété GPU du brief.

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
lora_config

## 9. `SFTConfig` — hyperparamètres de l'entraînement supervisé

`SFTConfig` (librairie **TRL**) hérite de `TrainingArguments` de Transformers
et y ajoute quelques options spécifiques au *supervised fine-tuning*
(notamment `dataset_text_field` et la gestion du *packing*).

| Paramètre | Valeur retenue | Rôle |
|---|---|---|
| `num_train_epochs` | **40** | Beaucoup d'epochs car le dataset est petit (< quelques centaines d'exemples) → on doit le voir plusieurs fois pour que les patterns métier soient bien capturés. |
| `per_device_train_batch_size` | **1** | Petit batch pour tenir sur une carte GPU modeste. |
| `gradient_accumulation_steps` | **4** | Batch *effectif* = `1 × 4 = 4`. On obtient un gradient plus stable sans saturer la VRAM. |
| `learning_rate` | **5e-4** | LR agressif assumé : LoRA tolère bien des LR plus élevés qu'un full fine-tuning (les paramètres entraînés sont peu nombreux et isolés). |
| `max_length` | **256** | Suffisant pour des tickets IT support (instruction + réponse courtes). |
| `report_to="none"` | — | **Confidentialité** : aucun tracker distant (W&B, etc.). |
| `push_to_hub=False` | — | **Confidentialité** : le modèle reste on-premise. |
| `dataset_text_field="text"` | — | Indique à `SFTTrainer` la colonne contenant le prompt déjà formaté. |


In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=40,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,
    max_length=256,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    push_to_hub=False,
    dataset_text_field="text",
    fp16=(DEVICE == "cuda"),
)
sft_config

## 10. `SFTTrainer` — orchestration & masquage de la loss

`SFTTrainer` (TRL) est un sur-ensemble du `Trainer` Hugging Face spécialisé
dans le **supervised fine-tuning** des LLM. Il prend en charge :

1. **L'injection du LoRA**
   Si on lui passe `peft_config=lora_config`, il applique automatiquement
   `get_peft_model(model, lora_config)` : seuls les paramètres LoRA seront
   entraînés (`requires_grad=True`), le reste est gelé.

2. **La préparation du dataset**
   La colonne `dataset_text_field` est tokenizée à la volée. Aucune
   pré-tokenisation manuelle n'est nécessaire.

3. **Le masquage de la loss sur le prompt — point crucial du brief**
   En *instruction tuning*, on veut que le modèle apprenne à **produire**
   la réponse, pas à **reproduire** l'instruction. `SFTTrainer` masque donc
   la partie *prompt* dans le calcul de la cross-entropy : les tokens situés
   avant le marqueur `### Réponse` sont étiquetés `-100` (valeur ignorée par
   `nn.CrossEntropyLoss`).
   Résultat : le gradient ne « pousse » que sur la portion réponse, ce qui
   évite que le modèle perde sa capacité générale en sur-apprenant le format
   de l'instruction.

4. **Le checkpointing & la sauvegarde**
   Compatible avec `save_strategy`, `save_total_limit`, etc.


In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds,
    peft_config=lora_config,
    processing_class=tokenizer,
)

# Vérifions le ratio de paramètres entraînables (« sobriété »)
trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in trainer.model.parameters())
print(f"Paramètres entraînables : {trainable:,} / {total:,} "
      f"({100*trainable/total:.2f} %)")

## 11. Entraînement

In [ ]:
trainer.train()

## 12. Sauvegarde (adapter LoRA + tokenizer)

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Modèle sauvegardé dans", OUTPUT_DIR)

## 13. Inférence APRÈS fine-tuning

In [ ]:
print(generate(trainer.model, QUESTION_TEST))

## 14. Test de généralisation
On vérifie le comportement sur une question **non vue** à l'entraînement.
Objectif : le modèle doit garder un format « AtosConnect / ticket / procédure »
sans inventer de procédure absurde.

In [ ]:
print(generate(trainer.model, QUESTION_GENERAL))